In [ ]:
import os
import numpy as np
import pandas as pd
import pydhn
from pydhn.fluids import Fluid
from pydhn.soils import Soil
from pydhn.solving import SimpleStep, Scheduler

# Parameters

In [ ]:
t_supply_producer = 70      # °C
delta_t_consumers = - 30    # K

rho_pipe = 940              # kg/m3
lambda_pipe = 0.35          # W/kg/k
cp_pipe = 2000              # J/kg/K
roughness_pipe = 0.007      # mm

rho_ins = 50                # kg/m3
lambda_ins = 0.026          # W/kg/k
cp_ins = 1500               # J/kg/K

# Network creation

In [ ]:
# Load the files with the DESTEST data
node_data = pd.read_csv("InputData/nodes_data.csv")
pipe_data = pd.read_csv("InputData/pipes_data.csv")

# Load csv data for heat profile at substations
profile_data = pd.read_csv("InputData/heat_profile_1_building_SFH_Network_1.csv")
profile_data = profile_data[:1008]
time = profile_data["Elapsed time [sec]"]
heat_demand = profile_data["Building heat demand [W]"]
NUM_SIM_STEPS = len(profile_data)
STEPSIZE = 600

# Change pipes direction to comply with pyDHN specifications
mapper = {"Beginning Node": "Ending Node", "Ending Node": "Beginning Node"}
pipe_data = pipe_data.rename(columns=mapper)

# Create network
net = pydhn.Network()

# Add nodes
for i in node_data.index:
    row = node_data.loc[i]
    name = row["Node"]
    name_s = name + "_s"
    name_r = name + "_r"
    X = row["X-Position [m]"]
    Y = row["Y-Position [m]"]
    net.add_node(name=name_s, x=X, y=Y)
    net.add_node(name=name_r, x=X, y=Y)
    # Add consumer
    if "SimpleDistrict" in name:
        net.add_consumer(
            name=name,
            start_node=name_s,
            end_node=name_r,
            control_type="energy",
            setpoint_type_hx="delta_t",
            setpoint_value_hx=delta_t_consumers,
            design_delta_t=abs(delta_t_consumers),
            t_out_min_hx=10,
            stepsize=STEPSIZE,
        )
    # Add producer
    if name == "i":
        net.add_producer(
            name="MAIN",
            start_node=name_r,
            end_node=name_s,
            setpoint_type_hx="t_out",
            setpoint_value_hx=t_supply_producer,
            stepsize=STEPSIZE,
        )

# Add pipes
for i in pipe_data.index:
    row = pipe_data.loc[i]
    start = row["Beginning Node"]
    end = row["Ending Node"]
    name_s = f"{start}_to_{end}_s"
    name_r = f"{start}_to_{end}_r"
    start_s = f"{start}_s"
    end_s = f"{end}_s"
    start_r = f"{end}_r"
    end_r = f"{start}_r"
    length = row["Length [m]"]
    diameter = row["Inner Diameter [m]"]
    thickness_ins = row["Insulation Thickness [m]"]
    thickness_pipe = float(row[" pipe_size"].split(" x ")[1]) / 1000

    net.add_lagrangian_pipe(
        name=name_s,
        start_node=start_s,
        end_node=end_s,
        diameter=diameter,
        length=length,
        roughness=roughness_pipe,
        k_insulation=lambda_ins,
        insulation_thickness=thickness_ins,
        k_internal_pipe=lambda_pipe,
        internal_pipe_thickness=thickness_pipe,
        casing_thickness=0.0,
        depth=1.5,
        rho_wall=rho_pipe,
        cp_wall=cp_pipe,
        stepsize=STEPSIZE,
        line="supply",
    )
    net.add_lagrangian_pipe(
        name=name_r,
        start_node=start_r,
        end_node=end_r,
        diameter=diameter,
        length=length,
        roughness=roughness_pipe,
        k_insulation=lambda_ins,
        insulation_thickness=thickness_ins,
        k_internal_pipe=lambda_pipe,
        internal_pipe_thickness=thickness_pipe,
        casing_thickness=0.0,
        depth=1.5,
        rho_wall=rho_pipe,
        cp_wall=cp_pipe,
        stepsize=STEPSIZE,
        line="return",
    )

# Create net, fluid and soil
fluid = Fluid(name="Water", isconstant=True, cp=4180, mu=0.0005434, rho=988, k=0.64)
soil = Soil(k=0.0, temp=10)

print(" --- Network setup complete. --- ")

# Simulation

In [ ]:
# --- Schedules definition ---

consumer_demand_schedule = pd.DataFrame()
deltat_consumers_schedule = pd.DataFrame()
consumer_names = []

# Add nodes / consumers / producer
for i in node_data.index:
    row = node_data.loc[i]
    name = row["Node"]

    # ---- Consumers ----
    if "SimpleDistrict" in name:
        consumer_names.append(name)
        consumer_demand_schedule[name] = - heat_demand
        deltat_consumers_schedule[name] = np.where(heat_demand == 0, np.nan, delta_t_consumers)

consumer_demand_schedule = consumer_demand_schedule[consumer_names]
deltat_consumers_schedule = deltat_consumers_schedule[consumer_names]

schedules = {
    "heat_demand": consumer_demand_schedule,
    #"setpoint_value_hx": deltat_consumers_schedule
}

In [ ]:
# --- Initializing the Network (Pre-heating) ---

print("--- Initializing Network (Pre-heating for stability) ---")

INIT_STEPS = 6  # (1 h)

init_demand_consumer = pd.DataFrame(
        np.tile(consumer_demand_schedule.iloc[0].values, (INIT_STEPS, 1)),
        columns=consumer_demand_schedule.columns
    )

init_deltat_consumer = pd.DataFrame(
        np.tile(deltat_consumers_schedule.iloc[0].values, (INIT_STEPS, 1)),
        columns=deltat_consumers_schedule.columns
    )

init_schedules = {
    "heat_demand": init_demand_consumer,
    #"setpoint_value_hx": init_deltat_consumer,
}

base_loop_init = SimpleStep(
    hydraulic_sim_kwargs={"error_threshold": 1, "verbose": False},
    thermal_sim_kwargs={"error_threshold": 1e-10, "verbose": False},
    with_thermal=True,
)

scheduler_init = Scheduler(
    base_loop=base_loop_init, schedules=init_schedules, steps=INIT_STEPS
)

# --- Timer per misurare costo computazionale ---
import time
start_time = time.perf_counter()

_ = scheduler_init.execute(net=net, fluid=fluid, soil=soil)

end_time = time.perf_counter()
sim_time = end_time - start_time

print(f"--- Network initialized over {INIT_STEPS} steps ({INIT_STEPS * 10 / 60} h). ---")
print(f"Tempo simulazione pre-heating pydhn: {sim_time:.4f} s")

In [ ]:
# --- Run simulation ---

print("--- Running with Scheduler (Full Simulation) ---")

base_loop_scheduler = SimpleStep(
    hydraulic_sim_kwargs={"error_threshold": 1, "verbose": False},
    thermal_sim_kwargs={"error_threshold": 1e-10, "verbose": False},
    with_thermal=True,
)

scheduler = Scheduler(
    base_loop=base_loop_scheduler,
    schedules=schedules,
    steps=NUM_SIM_STEPS,
)

# --- Timer per misurare tempo totale simulazione ---
start_time = time.perf_counter()

results_scheduler = scheduler.execute(net=net, fluid=fluid, soil=soil)

end_time = time.perf_counter()
sim_time_total = end_time - start_time

print("--- Simulation complete ---")
print(f"Average simulation time: {sim_time_total:.2f} s")

In [ ]:
# --- Get results ---
res = results_scheduler.to_dataframes()

t_supply_sd1 = res["nodes"]["temperature"]["SimpleDistrict_1_s"]
t_return_sd1 = res["nodes"]["temperature"]["SimpleDistrict_1_r"]
t_return_i = res["nodes"]["temperature"]["i_r"]

mdot_return_sd1 = res["edges"]["mass_flow"]["SimpleDistrict_1"]

qext_sd1 = mdot_return_sd1 * 4180 * (t_supply_sd1 - t_return_sd1)
qloss_supply_sd1 = - res["edges"]["delta_q"]["i_to_h_s"]
qloss_net = ((res["edges"]["delta_q"]["MAIN"]) - (-res["edges"]["delta_q"][consumer_names]).sum(axis=1))

dp_ds1 = res["edges"]["delta_p"]["SimpleDistrict_1"]
dp_return_i_h = res["edges"]["delta_p"]["i_to_h_r"]

res_to_save = pd.DataFrame({
    "Simple_District_1.supTemp.T|degC": t_supply_sd1,
    "Simple_District_1.retTemp.T|degC": t_return_sd1,
    "senTem_ret_i.T|degC": t_return_i,
    "Simple_District_1.plugFlowPipe1.port_a.m_flow|kg/s": mdot_return_sd1,
    "Simple_District_1.Q_flow_positive.y": qext_sd1,
    "Network_CE1_heatLoss[end].plugFlowPipe_ret24.heatPort.Q_flow*(-1)": qloss_supply_sd1,
    "fixedTemperature1.port.Q_flow|W": qloss_net,
    "Simple_District_1.hea.dp|Pa": dp_ds1,
    "Network_CE1_heatLoss[end].plugFlowPipe_ret24.cor.res.dp*(-1)": dp_return_i_h,
})

base_path = os.getcwd()
results_folder = os.path.join(base_path, 'Results')
results_file = os.path.join(results_folder, "network_CE1_pydhn_results.csv")
res_to_save.to_csv(results_file, index=False)

print(f"Results for CE1 implemented in pyDHN saved in: {results_file}")